# Batch Fermenter — Raw Construction

This notebook builds the same 0-D sparged batch fermenter as `batch_fermenter.ipynb`
but *without* the `FermenterBuilder` shortcut, exposing every layer the builder hides:

| Layer | Builder hides it | This notebook shows it |
|---|---|---|
| Species formulas + MWs | yes | yes |
| Reaction kinetics + stoichiometry | yes | yes |
| Acid-base equilibria | yes | yes |
| Gas-liquid transfer models + Henry constants | yes | yes |
| Gas / liquid phase construction | yes | yes |
| `ControlVolume` assembly | yes | yes |
| Boundary attachment | yes | yes |
| `Simulation` wiring | partially | yes |

Reading order: if you have not seen `batch_fermenter.ipynb`, start there for the
high-level pattern; come back here when you want to understand what is underneath.

## 1 · Environment setup

In [ ]:
import sys
from pathlib import Path

def _find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Run from inside the PyOMES repo")

_root = _find_root()
if str(_root / "models") not in sys.path:
    sys.path.insert(0, str(_root / "models"))

import numpy as np
import matplotlib.pyplot as plt

from PyOMES.chemistry import Species
from PyOMES.chemistry.databases.anaerobic_digestion import AD_BASIC
from PyOMES.reactions import (
    EquilibriumReaction, KineticReaction,
    ReactionBuilder, ReactionSystem,
)
from PyOMES.core import (
    ControlVolume, EquilibriumTransferModel, GasPhase, KineticTransferModel,
    LiquidPhase, Simulation,
)
from PyOMES.core.boundaries import GasFeed, PressureReliefVent
from PyOMES.core.phases import R_L_ATM_MOL_K
from PyOMES.control.cv_loops import PHController

print("Imports OK - repo root:", _root)

## 2 · Physical parameters

Henry partition constants are physical properties of each species — they live in
the chemistry database (`AD_BASIC`), not here. The parameters below are
*process* values that depend on the specific reactor design.

In [ ]:
T_K            = 305.15   # reactor temperature (K)
V_TOTAL_L      = 2.0      # total vessel volume (L)
HEADSPACE_FRAC = 0.20     # fraction of volume that is headspace

V_GAS = V_TOTAL_L * HEADSPACE_FRAC
V_LIQ = V_TOTAL_L * (1.0 - HEADSPACE_FRAC)

PH_SETPOINT = 5.0
TAU_H       = 5.0    # simulation duration (h)
N_STEPS     = 1000

# kLa values (1/h) — process parameters, not physical properties.
# CO2 gets 0.9 x kLa(O2) by the diffusivity-ratio convention.
KLA_PER_H = {"O2": 150.0, "CO2": 135.0}

print(f"V_gas = {V_GAS:.3f} L,  V_liq = {V_LIQ:.3f} L,  T = {T_K:.2f} K")

## 3 · Species declarations

Domain-specific species (substrate, biomass, organic acid) are declared
locally. Standard inorganics (H⁺, OH⁻, H₂O, HCO₃⁻, …) do not need to be
imported here — when stoichiometry is written as a string, the parser looks
them up automatically from `common_species`.

| Source | Species |
|---|---|
| `common_species` (auto) | H⁺, OH⁻, H₂O, HCO₃⁻ (resolved by string parser) |
| declared here | `ACETIC_ACID`, `ACETATE_MINUS`, `CO2`, `YEAST` |

`MW` is computed automatically from `atoms` using the IUPAC 2021 atomic weight
table unless overridden explicitly.  `YEAST` carries an explicit `MW=24.626`
because the empirical biomass formula used here (CH₁.₆₁O₀.₅₆) omits nitrogen,
so the atoms-derived value would underestimate the true formula unit mass.

In [ ]:
ACETIC_ACID   = Species(id="AceticAcid",  atoms={"C":2,"H":4,"O":2},       charge=0)
ACETATE_MINUS = Species(id="Acetate-",    atoms={"C":2,"H":3,"O":2},       charge=-1)
CO2           = Species(id="CO2",         atoms={"C":1,"O":2},              charge=0)
YEAST         = Species(id="Yeast",       atoms={"C":1,"H":1.61,"O":0.56}, charge=0, MW=24.626)

print("Locally declared species (MW auto-computed except Yeast):")
for sp in [ACETIC_ACID, ACETATE_MINUS, CO2, YEAST]:
    print(f"  {sp.id:>14}  MW={float(sp.MW):.3f}  charge={sp.charge}")

## 4 · Kinetic reaction: aerobic growth on acetic acid

`ReactionBuilder.monod_aerobic_growth` takes the species objects and three
kinetic parameters and returns a fully stoichiometrically balanced
`KineticReaction` — it builds the Monod rate closure and calls
`ReactionBuilder.aerobic_growth` internally.

The rate law (extensive, mol substrate consumed / h):

$$
r = \frac{\mu_{\max} \cdot S_{g/L}}{K_S + S_{g/L}}
    \cdot \frac{X_{g/L}}{Y}
    \cdot \frac{V_L}{\text{MW}_S}
$$

Stoichiometric coefficients for O₂, CO₂, and H₂O are derived automatically
from elemental balance — you supply the formulas and the yield; the
stoichiometry falls out.

In [ ]:
MU_MAX = 0.5    # 1/h — maximum specific growth rate
KS_G_L = 5e-3  # g/L — substrate half-saturation constant
YIELD  = 0.36   # g biomass / g substrate

rxn_growth = ReactionBuilder.monod_aerobic_growth(
    substrate    = ACETIC_ACID,
    biomass      = YEAST,
    mu_max_per_h = MU_MAX,
    Ks_gL        = KS_G_L,
    yield_gX_gS  = YIELD,
    label        = "growth_on_AceticAcid",
)

print("Kinetic reaction:", rxn_growth.label)
print("Stoichiometry:")
for e in rxn_growth.stoichiometry:
    print(f"  {e.coefficient:+.4g}  {e.species.id}  ({e.phase})")

## 5 · Equilibrium reactions

Three reactions make up the aqueous chemistry, split into three cells below.
The first two (water and carbonate) are the thermodynamic backbone needed for
correct pH and CO₂ speciation. The third is the organic acid.

| Sub-section | Reaction | Type |
|---|---|---|
| 5a | H₂O ⇌ H⁺ + OH⁻ | single-phase (water) |
| 5b | Carbonate ladder: CO₂ + H₂O ⇌ HCO₃⁻ + H⁺ ; HCO₃⁻ ⇌ H⁺ + CO₃²⁻ | single-phase |
| 5c | AceticAcid ⇌ Acetate⁻ + H⁺ | single-phase |

Stoichiometry is written in chemical notation (`"reactants <-> products"`).
The parser resolves standard inorganics automatically from `common_species`;
locally declared species (like `AceticAcid`) must be passed via
`species={...}`.

No cross-phase routing reaction is needed for CO₂: the `transfer_models` kwarg
seeds `speciation_keys["CO2"] = "CO2"` automatically at CV construction, so
the BFS for the CO₂ speciation ladder runs correctly without any extra
boilerplate declaration.

### 5a — Water dissociation

H₂O ⇌ H⁺ + OH⁻. `pKw = 14.0` at 25 °C; `dH = 55 900 J/mol` (van 't Hoff
correction). Without this the speciation engine falls back to a hardcoded
`pKw = 14` default — including it explicitly makes the assumption visible and
allows overriding for non-standard conditions.

In [ ]:
rxn_water = EquilibriumReaction(
    "H2O,aq <-> H+,aq + OH-,aq",
    log_K=-14.0,
    dH_J_per_mol=55900.0,
    T_ref_K=298.15,
    label="eq_water",
)
print(rxn_water.label, f"  log_K = {rxn_water.log_K}")

### 5b — Carbonate ladder

Two coupled equilibria span the full inorganic carbon system.

**pKa1 = 6.35** — CO₂(aq) + H₂O ⇌ HCO₃⁻ + H⁺. Seeds the CO₂ speciation
ladder in `derive_speciation_keys`: the BFS from dissolved CO₂ through
single-phase equilibria finds HCO₃⁻ here, enabling the gas-liquid link to
compute the α-correction (fraction of dissolved inorganic carbon that is free
molecular CO₂). Without this, the link always sees α = 1 and overstates the
CO₂ transfer driving force at any pH where bicarbonate is significant.

**pKa2 = 10.33** — HCO₃⁻ ⇌ H⁺ + CO₃²⁻. Completes the ladder so the
speciation engine can compute CO₃²⁻ at high pH. In this fermenter (pH target
5.0) CO₃²⁻ will be negligible, but including the step is correct chemistry and
costs nothing at runtime. Both species are in `common_species` — no `species`
kwarg needed.

In [ ]:
rxn_co2_aq = EquilibriumReaction(
    "CO2,aq + H2O,aq <-> HCO3-,aq + H+,aq",
    log_K=-6.35,
    dH_J_per_mol=7646.0,
    T_ref_K=298.15,
    total_id="CO2",
    label="eq_CO2",
)
print(rxn_co2_aq.label, f"  log_K = {rxn_co2_aq.log_K}")

In [ ]:
rxn_co3 = EquilibriumReaction(
    "HCO3-,aq <-> H+,aq + CO3--,aq",
    log_K=-10.33,
    label="eq_HCO3",
)
print(rxn_co3.label, f"  log_K = {rxn_co3.log_K}")

### 5c — Acetic acid dissociation

AceticAcid ⇌ Acetate⁻ + H⁺, `pKa = 4.756`.

In [ ]:
rxn_dissoc = EquilibriumReaction(
    "AceticAcid,aq <-> Acetate-,aq + H+,aq",
    species={"AceticAcid": ACETIC_ACID, "Acetate-": ACETATE_MINUS},
    log_K=-4.756,
    label="eq_AceticAcid",
)
print(rxn_dissoc.label, f"  log_K = {rxn_dissoc.log_K}")

### 5d — Mass balance check

Each reaction exposes `.show_balance()`, which prints the net elemental
residual for every element present across the participants.  No element list
needed — the method discovers them from the species atoms automatically.
Pass `elements=(...)` to restrict the check to a subset.

In [ ]:
for rxn in [rxn_water, rxn_co2_aq, rxn_co3, rxn_dissoc]:
    rxn.show_balance()
    print()

### 5e — Van 't Hoff temperature dependence

Each `EquilibriumReaction` exposes `.plot_vant_hoff()`, which uses the stored
`log_K`, `dH_J_per_mol`, and `T_ref_K` to draw the temperature dependence of
the equilibrium constant via the Van 't Hoff equation:

$$
\log_{10} K(T) = \log_{10} K(T_{\mathrm{ref}})
    + \frac{\Delta H^\circ}{R \ln 10}
      \left(\frac{1}{T_{\mathrm{ref}}} - \frac{1}{T}\right)
$$

The primary axis shows $\log_{10} K$; a secondary axis shows
$\mathrm{p}K_a = -\log_{10} K$.  The reference point $(T_{\mathrm{ref}},
\log_{10} K)$ is marked.  Reactions without `dH_J_per_mol` draw a flat line
with an annotation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

rxn_water.plot_vant_hoff(T_range_K=(273.15, 373.15), ax=axes[0])
rxn_co2_aq.plot_vant_hoff(T_range_K=(273.15, 373.15), ax=axes[1])

fig.suptitle("Van 't Hoff temperature dependence", fontsize=12)
plt.tight_layout()
plt.show()

## 6 · Assemble the ReactionSystem

`ReactionSystem` accepts a flat list of reactions and *pre-buckets* them by type
at construction. The `ControlVolume` later routes each bucket to the correct
solver: kinetic reactions to the ODE integrator, single-phase equilibria to the
speciation engine.

The full set here is five reactions — no cross-phase routing declaration is
needed because the `transfer_models` kwarg seeds `speciation_keys`
automatically from the partition models at CV construction.

| Reaction | Type | Solver |
|---|---|---|
| `rxn_growth` | kinetic | ODE integrator |
| `rxn_water` | single-phase equilibrium | speciation engine |
| `rxn_co2_aq` | single-phase equilibrium | speciation engine (seeds CO₂ ladder) |
| `rxn_dissoc` | single-phase equilibrium | speciation engine |
| `rxn_co3` | single-phase equilibrium | speciation engine (second carbonate step) |

In [ ]:
rxn_system = ReactionSystem(
    [rxn_growth, rxn_water, rxn_co2_aq, rxn_dissoc, rxn_co3],
    label="raw_notebook_chemistry",
)

print(f"Kinetic reactions      : {len(rxn_system.kinetic_reactions)}")
print(f"Single-phase equilibria: {len(rxn_system.single_phase_equilibria)}")
print(f"  -> ", [r.label for r in rxn_system.single_phase_equilibria])
print(f"Cross-phase equilibria : {len(rxn_system.cross_phase_equilibria)}")
print(f"Species in system      : {', '.join(rxn_system.species_ids)}")

print("\nMass balance across all reactions:")
print("-" * 40)
rxn_system.show_balance()

### 6b — Speciation diagrams

`rxn_system.plot_speciation(anchor_id)` scans `single_phase_equilibria` for
reactions of the form `acid ⇌ base + H⁺`, walks the connected chain from
*anchor_id* outward, and plots the molar fraction α of each form against pH.
The pKa crossover points are annotated automatically.

The anchor can be **any** species in the ladder — the method finds the
most-protonated root and orders the chain from there.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Carbonate ladder: CO2 → HCO3- → CO3--  (two pKa steps)
rxn_system.plot_speciation("CO2", ax=axes[0])

# Acetic acid ladder: AceticAcid → Acetate-  (one pKa step)
rxn_system.plot_speciation("AceticAcid", ax=axes[1])

fig.suptitle("Speciation diagrams from ReactionSystem", fontsize=12)
plt.tight_layout()
plt.show()

## 7 · Build the gas phase

`GasPhase` stores mole counts for each gas species. We seed it with air at 1 atm
using the ideal gas law: n = PV / RT, then split by volume fractions.

In [ ]:
n_total_gas = (1.0 * V_GAS) / (R_L_ATM_MOL_K * T_K)  # PV = nRT

gas = GasPhase(
    n_mol={
        "O2":  n_total_gas * 0.2095,
        "CO2": n_total_gas * 0.0004,
        "N2":  n_total_gas * 0.7901,
    },
    V_L=V_GAS,
    T_K=T_K,
)

print(f"Total gas moles: {n_total_gas:.4f}")
for sp, n in gas.n_mol.items():
    print(f"  {sp:>4}: {n:.4e} mol")

## 8 · Build the liquid phase

Initial dissolved-gas concentrations come from Henry equilibrium with the headspace:
n_dissolved = H · p_partial · V_liq.  Substrate and biomass are set from
desired mass concentrations. `H+` gets a placeholder that the speciation engine
overwrites on the first step.

In [ ]:
p_atm = gas.p_atm   # {species: partial pressure (atm)}

# Substrate: 1.2 g/L acetic acid total acetate
n_acetate = (1.2 / float(ACETIC_ACID.MW)) * V_LIQ
# Inoculum: 0.1 g/L yeast
n_yeast   = (0.1 / float(YEAST.MW))       * V_LIQ

# Initial dissolved gases from Henry equilibrium with the headspace.
# H_ref is in mol/(m3*Pa); convert: mol/m3 * (1 m3 / 1000 L) * V_LIQ L
def _henry_n(sp):
    pm = AD_BASIC.partition_models[sp]
    return pm.H_ref * p_atm.get(sp, 0.0) * 101325 / 1000 * V_LIQ

liquid = LiquidPhase(
    n_mol={
        ACETIC_ACID.id:   n_acetate,
        ACETATE_MINUS.id: 0.0,   # speciation engine fills this
        YEAST.id:         n_yeast,
        "O2":             _henry_n("O2"),
        "CO2":            _henry_n("CO2"),
        "N2":             _henry_n("N2"),
        "HCO3-":          0.0,   # speciation engine fills this (carbonate ladder)
        "CO3--":          0.0,   # speciation engine fills this (second carbonate step)
        "OH-":            0.0,   # speciation engine fills this (water dissociation)
        "H+":             1.0e-7 * V_LIQ,  # placeholder; engine overwrites
    },
    V_L=V_LIQ,
    T_K=T_K,
)

print("Initial liquid inventory (mol):")
for sp, n in liquid.n_mol.items():
    print(f"  {sp:>14}: {n:.4e}")

## 9 · Declare the transfer models

`AD_BASIC.partition_models` is a dict of pre-built, temperature-corrected
`HenryPartition` objects — one per gas species the database knows about.
Each stores `H_ref` (mol/(m³·Pa) at `T_ref_K`) and `dlnH` (van 't Hoff slope
in K), so solubility is re-evaluated at the reactor temperature automatically.

`kLa` values are *process* parameters (reactor geometry, agitator speed, etc.)
that the database cannot supply — they are specified here.

| Type | Species | Meaning |
|---|---|---|
| `KineticTransferModel` | O₂ | rate-limited absorption; driving force on total dissolved pool |
| `KineticTransferModel` | CO₂ | rate-limited; `transfer_basis="molecular"` — driving force on free molecular CO₂ only, matching BSM2 kinetics |
| `EquilibriumTransferModel` | N₂ | instantaneous Henry equilibrium each step (largely inert at this timescale) |

`transfer_basis="molecular"` for CO₂ means α (the fraction of dissolved
inorganic carbon that is free CO₂) is applied to the liquid-side concentration
rather than to the effective Henry constant. This matches the BSM2 convention
and avoids overstating the transfer driving force at elevated pH where HCO₃⁻
dominates.

In [ ]:
print("All partition models in AD_BASIC:")
for sp, pm in AD_BASIC.partition_models.items():
    print(f"  {sp:>4}  H_ref={pm.H_ref:.2e} mol/(m3*Pa),  dlnH={pm.dlnH:.0f} K")

transfer_models = {
    "O2":  KineticTransferModel(
               AD_BASIC.partition_models["O2"],
               k_transfer=KLA_PER_H["O2"],
           ),
    "CO2": KineticTransferModel(
               AD_BASIC.partition_models["CO2"],
               k_transfer=KLA_PER_H["CO2"],
               transfer_basis="molecular",
           ),
    "N2":  EquilibriumTransferModel(
               AD_BASIC.partition_models["N2"],
           ),
}

print("\nTransfer models:")
for sp, tm in transfer_models.items():
    print(f"  {sp:>4}: {tm}")

## 10 · Assemble the ControlVolume

`ControlVolume` binds phases, transfer models, and the reaction system
together. Passing `transfer_models` causes the CV to auto-detect the
`GasPhase`/`LiquidPhase` pair, build an internal gas-liquid link, and seed
its `speciation_keys` mapping. For every species in `transfer_models`, an
identity entry (`sp → sp`) is inserted automatically, so the BFS for
acid-base ladders (e.g. the CO₂ / HCO₃⁻ ladder from `rxn_co2_aq`) runs
without any extra cross-phase routing declarations.

The `transfer_models` dict is stored on `cv.transfer_models` for later
inspection (e.g. reading `k_transfer` values post-construction).

In [ ]:
cv = ControlVolume(
    phases          = {"gas": gas, "liquid": liquid},
    transfer_models = transfer_models,
    reaction_system = rxn_system,
    label           = "raw_notebook",
)

print("CV label:", cv.label)
print("Phases:  ", list(cv.phases.keys()))
print("Transfer models:", list(cv.transfer_models.keys()))

# k_transfer values are readable from cv.transfer_models post-construction
for sp, tm in cv.transfer_models.items():
    kLa = getattr(tm, "k_transfer", "—")
    basis = getattr(tm, "transfer_basis", "equilibrium")
    print(f"  {sp:>4}  kLa={kLa}  basis={basis}")

## 11 · Attach boundaries and build the controller

Boundaries are appended to `cv.boundaries` after construction — the same pattern
the builder uses internally.

- **`GasFeed`** — adds sparge gas at `vvm_min` volumes of liquid per minute.
- **`PressureReliefVent`** — opens when headspace pressure exceeds `P_set_atm`,
  venting gas back to the set-point pressure instantly.
- **`PHController`** — doses acid (`H3PO4`) or base (`NaOH`) to hold pH at the
  setpoint. `Kp` and `Ki` are the proportional and integral gains.

In [ ]:
cv.boundaries.append(GasFeed(
    vvm_min          = 1.0,
    y                = {"O2": 0.21, "N2": 0.79},
    P_inlet_atm      = 1.0,
    phase_key        = "gas",
    liquid_phase_key = "liquid",
    label            = "air_sparge",
))
cv.boundaries.append(PressureReliefVent(P_set_atm=1.10, mode="instant"))

ph_ctrl = PHController(
    setpoint         = PH_SETPOINT,
    Kp               = 0.5,
    Ki               = 0.0,
    chemical_id      = "H3PO4",
    base_chemical_id = "NaOH",
    max_add_molL_hr  = 0.05,
)

print("Boundaries:", [b.label for b in cv.boundaries])
print("pH controller setpoint:", ph_ctrl.setpoint)

## 12 · Build the Simulation

`Simulation` owns the CV map and the list of controllers. On `sim.run()`, it
steps each controller once per output interval and advances each CV through the
full operator-splitting cycle (transfer → reaction → speciation).

In [ ]:
sim = Simulation(
    cvs         = {"main": cv},
    controllers = [ph_ctrl],
    label       = "raw_notebook_sim",
)

print("Simulation:", sim.label)
print("CVs:       ", list(sim.cvs.keys()))

## 13 · Run

In [ ]:
result = sim.run(tau_h=TAU_H, n_steps=N_STEPS)
print(f"Finished in {result.runtime_s:.2f} s")

## 14 · Results summary

`liquid_mol[cv_key][species_id]` is an ndarray of shape `(N_STEPS+1,)`.
Index `[0]` is the initial state; `[-1]` is the final state.

In [ ]:
cv = "main"

print(f"t = 0 to {TAU_H} h  |  {N_STEPS} steps  |  pH setpoint = {PH_SETPOINT}")
print("\nInitial -> Final liquid inventory (mol):")
for sp in sorted(result.liquid_mol[cv]):
    n0 = result.liquid_mol[cv][sp][0]
    nf = result.liquid_mol[cv][sp][-1]
    print(f"  {sp:>14}: {n0:>10.4e}  ->  {nf:>10.4e}")

pH = result.pH[cv]
pH_valid = pH[np.isfinite(pH)]
print(f"\npH: {pH_valid[0]:.3f}  ->  {pH_valid[-1]:.3f}")
print(f"Runtime: {result.runtime_s:.3f} s")

## 15 · Time-series plots

In [ ]:
cv  = "main"
t   = result.t_h
liq = result.liquid_mol[cv]
pH  = result.pH[cv]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Batch Fermenter — raw construction", fontsize=13)

ax = axes[0, 0]
ax.plot(t, liq.get("AceticAcid", np.zeros_like(t)), color="tab:orange")
ax.set(xlabel="Time (h)", ylabel="mol", title="Substrate (AceticAcid)")

ax = axes[0, 1]
ax.plot(t, liq.get("Yeast", np.zeros_like(t)), color="tab:green")
ax.set(xlabel="Time (h)", ylabel="mol", title="Biomass (Yeast)")

ax = axes[1, 0]
ax.plot(t, liq.get("O2", np.zeros_like(t)), color="tab:blue")
ax.set(xlabel="Time (h)", ylabel="mol", title="Dissolved Oxygen (liquid O2)")

ax = axes[1, 1]
ax.plot(t, pH, color="tab:red", label="pH")
ax.axhline(PH_SETPOINT, ls="--", color="gray", label=f"setpoint ({PH_SETPOINT})")
ax.set(xlabel="Time (h)", ylabel="pH", title="pH")
ax.legend()

plt.tight_layout()
plt.show()